<a href="https://colab.research.google.com/github/nguyenvannguyen1125/nguyen604/blob/main/nguyen233030604.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

# Quan hệ R(ID, Name, Age)
R_data = [(1, "An", 22), (2, "Binh", 19), (3, "Cuong", 25)]
R_rdd = sc.parallelize(R_data)

# Điều kiện chọn C: Age > 20
# Giai đoạn Map: Kiểm tra điều kiện C và tạo cặp Key-Value (t, t)
mapped_rdd = R_rdd.flatMap(lambda t: [(t, t)] if t[2] > 20 else [])

# Giai đoạn Reduce / Action: Lấy danh sách các bản ghi thỏa điều kiện
result = mapped_rdd.keys().collect()
print("Selection (Age > 20):", result)

Selection (Age > 20): [(1, 'An', 22), (3, 'Cuong', 25)]


In [ ]:
# Quan hệ R(ID, Department, Role) có dữ liệu trùng thuộc tính chiếu
R_data = [(1, "IT", "Dev"), (2, "IT", "Dev"), (3, "HR", "Recruiter")]
R_rdd = sc.parallelize(R_data)

# Chiếu lên S = {Department, Role}
# 1. Map: Bỏ ID, tạo t' = (Department, Role), phát ra (t', t')
mapped_rdd = R_rdd.map(lambda t: ((t[1], t[2]), (t[1], t[2])))

# 2. Reduce: Gom nhóm theo Key t' và loại bỏ trùng lặp
reduced_rdd = mapped_rdd.reduceByKey(lambda v1, v2: v1)

result = reduced_rdd.keys().collect()
print("Projection (Department, Role):", result)


Projection (Department, Role): [('IT', 'Dev'), ('HR', 'Recruiter')]


In [ ]:
R_rdd = sc.parallelize([("a", 1), ("b", 2)])
S_rdd = sc.parallelize([("b", 2), ("c", 3)])

# Map & Union: Tạo cặp (t, t) và gộp 2 RDD
union_rdd = R_rdd.union(S_rdd).map(lambda t: (t, t))

# Reduce: Bỏ trùng lặp bằng reduceByKey
result_union = union_rdd.reduceByKey(lambda x, y: x).keys().collect()
print("Union R U S:", result_union)

Union R U S: [('c', 3), ('a', 1), ('b', 2)]


In [ ]:
R_rdd = sc.parallelize([("a", 1), ("b", 2)])
S_rdd = sc.parallelize([("b", 2), ("c", 3)])
# Map: Gắn nhãn nguồn hoặc giữ nguyên Key (t, t)
combined_rdd = R_rdd.map(lambda t: (t, t)).union(S_rdd.map(lambda t: (t, t)))

# Reduce: Chỉ giữ lại các Key có từ 2 Value trở lên (xuất hiện ở cả R và S)
grouped_rdd = combined_rdd.groupByKey()
intersection_rdd = grouped_rdd.filter(lambda x: len(list(x[1])) >= 2).keys()

print("Intersection R n S:", intersection_rdd.collect())

Intersection R n S: [('b', 2)]


In [ ]:
# Map: Phát ra (t, 'R') cho R và (t, 'S') cho S
R_tagged = R_rdd.map(lambda t: (t, 'R'))
S_tagged = S_rdd.map(lambda t: (t, 'S'))

# Shuffle & Reduce: Group theo Key t và kiểm tra danh sách nguồn
difference_rdd = R_tagged.union(S_tagged) \
    .groupByKey() \
    .filter(lambda x: list(x[1]) == ['R']) \
    .keys()

print("Difference R - S:", difference_rdd.collect())
# Kết quả: [('a', 1)]

Difference R - S: [('a', 1)]


In [ ]:
# R(A, B) và S(B, C) với B là thuộc tính chung
R_rel = sc.parallelize([(101, "B1"), (102, "B2")])    # (A, B)
S_rel = sc.parallelize([("B1", "X"), ("B1", "Y")])    # (B, C)

# 1. Map Phase: Tạo key là B, value chứa nhãn nguồn và thuộc tính còn lại
R_mapped = R_rel.map(lambda t: (t[1], ('R', t[0])))    # (B, ('R', A))
S_mapped = S_rel.map(lambda t: (t[0], ('S', t[1])))    # (B, ('S', C))

# 2. Shuffle Phase: Gom các bản ghi có cùng Key B
joined_input = R_mapped.union(S_mapped).groupByKey()

# 3. Reduce Phase: Tách danh sách R và S, ghép cặp Cartesian
def reduce_join(item):
    b_key, values = item
    r_vals = [val[1] for val in values if val[0] == 'R']
    s_vals = [val[1] for val in values if val[0] == 'S']

    # Tạo tích Cartesian giữa r_vals và s_vals
    res = []
    for a in r_vals:
        for c in s_vals:
            res.append((a, b_key, c))
    return res
#flatMap giúp gộp 2 key
result_join = joined_input.flatMap(reduce_join).collect()
print("Natural Join R <>< S:", result_join)
# Kết quả: [(101, 'B1', 'X'), (101, 'B1', 'Y')]

Natural Join R <>< S: [(101, 'B1', 'X'), (101, 'B1', 'Y')]


bài 1 đếm từ

In [1]:
from pyspark import SparkContext, SparkConf

# 1. Khởi tạo SparkContext
conf = SparkConf().setAppName("WordCountMapReduce").setMaster("local[*]")
sc = SparkContext(conf=conf)

# 2. Tạo tập dữ liệu đầu vào phân tán (RDD)
data = [
    "Học lập trình PySpark rất thú vị",
    "PySpark giúp xử lý dữ liệu lớn Big Data",
    "Lập trình MapReduce với PySpark hiệu quả"
]

rdd = sc.parallelize(data)

# 3. Thực hiện quy trình MapReduce
word_counts = (
    rdd.flatMap(lambda line: line.split(" "))       # Giai đoạn Map 1: Tách dòng thành các từ
       .map(lambda word: (word.lower(), 1))         # Giai đoạn Map 2: Gán cặp (từ, 1)
       .reduceByKey(lambda a, b: a + b)             # Giai đoạn Reduce: Gom nhóm và cộng dồn theo từ
)

# 4. Thu thập và hiển thị kết quả
for word, count in word_counts.collect():
    print(f"{word}: {count}")

# Dừng SparkContext
sc.stop()

trình: 2
thú: 1
vị: 1
xử: 1
lý: 1
dữ: 1
liệu: 1
lớn: 1
big: 1
quả: 1
học: 1
lập: 2
pyspark: 3
rất: 1
giúp: 1
data: 1
mapreduce: 1
với: 1
hiệu: 1


bài 2


In [2]:
from pyspark import SparkContext, SparkConf

# Khởi tạo Spark
conf = SparkConf().setAppName("MatrixVectorMultiplication").setMaster("local[*]")
sc = SparkContext(conf=conf)

# 1. Dữ liệu đầu vào

# Ma trận M định dạng theo khóa cột:
# (j, (i, M_ij))
matrix_by_col = sc.parallelize([
    (0, (0, 1.0)),       # Cột 0, Hàng 0, Giá trị 1.0
    (1, (0, 2.0)),       # Cột 1, Hàng 0, Giá trị 2.0
    (0, (1, 3.0)),       # Cột 0, Hàng 1, Giá trị 3.0
    (1, (1, 4.0))        # Cột 1, Hàng 1, Giá trị 4.0
])

# Vector v định dạng: (j, v_j)
# Vector v = [5.0, 10.0]^T
vector_rdd = sc.parallelize([
    (0, 5.0),             # Chỉ số 0, Giá trị 5.0
    (1, 10.0)             # Chỉ số 1, Giá trị 10.0
])

# 2. Giai đoạn MapReduce

# Bước a: Join theo khóa cột j
# -> Kết quả: (j, ((i, M_ij), v_j))
joined_rdd = matrix_by_col.join(vector_rdd)

# Bước b: Tính tích từng phần và đổi khóa thành hàng i
# -> Kết quả: (i, M_ij * v_j)
products_rdd = joined_rdd.map(
    lambda x: (
        x[1][0][0],
        x[1][0][1] * x[1][1]
    )
)

# Bước c: Reduce để cộng dồn theo hàng i
# -> x_i = sum(M_ij * v_j)
result_vector = products_rdd.reduceByKey(
    lambda a, b: a + b
)

# 3. Thu thập kết quả
for row_idx, val in sorted(result_vector.collect()):
    print(f"x[{row_idx}] = {val}")

# Dừng SparkContext
sc.stop()

x[0] = 25.0
x[1] = 55.0


In [3]:
# Khởi tạo Spark
conf = SparkConf().setAppName("MatrixMultiplication_4x5_5x4").setMaster("local[*]")
sc = SparkContext(conf=conf)

# 1. Dữ liệu đầu vào
# Ma trận A kích thước 4x5 (i từ 0..3, k từ 0..4)
# Định dạng theo khóa trung gian k: (k, ('A', i, A_ik))
matrix_A_data = [
    # k = 0
    (0, ('A', 0, 1.0)), (0, ('A', 1, 2.0)), (0, ('A', 2, 3.0)), (0, ('A', 3, 4.0)),
    # k = 1
    (1, ('A', 0, 5.0)), (1, ('A', 1, 1.0)), (1, ('A', 2, 2.0)), (1, ('A', 3, 3.0)),
    # k = 2
    (2, ('A', 0, 1.0)), (2, ('A', 1, 1.0)), (2, ('A', 2, 1.0)), (2, ('A', 3, 1.0)),
    # k = 3
    (3, ('A', 0, 2.0)), (3, ('A', 1, 0.0)), (3, ('A', 2, 1.0)), (3, ('A', 3, 2.0)),
    # k = 4
    (4, ('A', 0, 1.0)), (4, ('A', 1, 2.0)), (4, ('A', 2, 0.0)), (4, ('A', 3, 1.0)),
]
rdd_A = sc.parallelize(matrix_A_data)

# Ma trận B kích thước 5x4 (k từ 0..4, j từ 0..3)
# Định dạng theo khóa trung gian k: (k, ('B', j, B_kj))
matrix_B_data = [
    # k = 0
    (0, ('B', 0, 1.0)), (0, ('B', 1, 0.0)), (0, ('B', 2, 2.0)), (0, ('B', 3, 1.0)),
    # k = 1
    (1, ('B', 0, 2.0)), (1, ('B', 1, 1.0)), (1, ('B', 2, 0.0)), (1, ('B', 3, 1.0)),
    # k = 2
    # k = 2
    (2, ('B', 0, 0.0)), (2, ('B', 1, 2.0)), (2, ('B', 2, 1.0)), (2, ('B', 3, 0.0)),
    # k = 3
    (3, ('B', 0, 1.0)), (3, ('B', 1, 1.0)), (3, ('B', 2, 1.0)), (3, ('B', 3, 1.0)),
    # k = 4
    (4, ('B', 0, 3.0)), (4, ('B', 1, 0.0)), (4, ('B', 2, 1.0)), (4, ('B', 3, 2.0)),
]
rdd_B = sc.parallelize(matrix_B_data)

# 2. Giai đoạn MapReduce:
# Bước a: Join theo khóa trung gian k -> Kết quả: (k, (('A', i, A_ik), ('B', j, B_kj)))
joined_rdd = rdd_A.join(rdd_B)

# Bước b: Tính tích từng phần và đổi khóa thành toạ độ (i, j) của ma trận kết quả C
# x[1][0] là tuple của A: ('A', i, A_ik) -> lấy i ở index 1, A_ik ở index 2
# x[1][1] là tuple của B: ('B', j, B_kj) -> lấy j ở index 1, B_kj ở index 2
products_rdd = joined_rdd.map(lambda x:
    ((x[1][0][1], x[1][1][1]), x[1][0][2] * x[1][1][2])
)

# Bước c: Reduce để cộng dồn theo toạ độ (i, j) -> C_ij = sum(A_ik * B_kj)

result_matrix = products_rdd.reduceByKey(lambda a, b: a + b)

# 3. Thu thập và hiển thị kết quả ma trận C (kích thước 4x4)
print("Kết quả phép nhân ma trận A (4x5) với ma trận B (5x4) -> Ma trận C (4x4):")
for (i, j), val in sorted(result_matrix.collect()):
    print(f"C[{i}][{j}] = {val}")

# Dừng SparkContext
sc.stop()

Kết quả phép nhân ma trận A (4x5) với ma trận B (5x4) -> Ma trận C (4x4):
C[0][0] = 16.0
C[0][1] = 9.0
C[0][2] = 6.0
C[0][3] = 10.0
C[1][0] = 10.0
C[1][1] = 3.0
C[1][2] = 7.0
C[1][3] = 7.0
C[2][0] = 8.0
C[2][1] = 5.0
C[2][2] = 8.0
C[2][3] = 6.0
C[3][0] = 15.0
C[3][1] = 7.0
C[3][2] = 12.0
C[3][3] = 11.0


In [4]:
from pyspark import SparkContext, SparkConf

# Khởi tạo Spark
conf = SparkConf().setAppName("PageRankWithoutDamping").setMaster("local[*]")
sc = SparkContext(conf=conf)

# 1. Dữ liệu đồ thị đầu vào (danh sách các cạnh: node, neighbor)
# Ví dụ đồ thị gồm 4 trang: 0, 1, 2, 3
edges_data = [
    (0, 1), (0, 2),   # Trang 0 trỏ tới 1 và 2
    (1, 2),           # Trang 1 trỏ tới 2
    (2, 0),           # Trang 2 trỏ tới 0
    (3, 2),           # Trang 3 trỏ tới 2
]
edges_rdd = sc.parallelize(edges_data)

# Tiền xử lý: Nhóm các cạnh theo từng node nguồn để tạo danh sách các trang kề (adjacency list)
# Kết quả: (node, [neighbors])
links_rdd = edges_rdd.groupByKey().cache()

# Danh sách toàn bộ node trong đồ thị (dùng để chống mất node không có ai trỏ tới)
all_nodes_rdd = links_rdd.map(lambda x: x[0]).cache()
num_nodes = all_nodes_rdd.count()

# 2. Khởi tạo PageRank ban đầu cho mỗi node bằng 1.0/N
# Kết quả: (node, rank)
ranks_rdd = links_rdd.map(lambda x: (x[0], 1.0 / num_nodes))

# Số lần lặp (iterations)
num_iterations = 10

for iteration in range(num_iterations):
    # Bước a: Kết hợp danh sách liên kết với giá trị PageRank hiện tại
    # joins_rdd dạng: (node, ([neighbors], rank))
    joined_rdd = links_rdd.join(ranks_rdd)

    # Bước b: Phân phối PageRank từ node nguồn tới tất cả các neighbors của nó
    # flatMap sinh ra các cặp (neighbor node, contribution)
    contributions_rdd = joined_rdd.flatMap(lambda x:
        [(neighbor, x[1][1] / len(x[1][0])) for neighbor in x[1][0]]
    )

    # Chống mất node: union thêm các node có rank=0 để node không ai trỏ tới
    # (như node 3 trong ví dụ) vẫn xuất hiện trong kết quả sau reduceByKey
    contributions_rdd = contributions_rdd.union(
        all_nodes_rdd.map(lambda n: (n, 0.0))
    )

    # Bước c: Cộng dồn các đóng góp cho từng node để tính PageRank mới
    ranks_rdd = contributions_rdd.reduceByKey(lambda a, b: a + b)

# 3. Thu thập và hiển thị kết quả PageRank sau khi hội tụ (không có damping factor)
print(f"Kết quả PageRank sau {num_iterations} vòng lặp (không có damping factor):")
for node, rank in sorted(ranks_rdd.collect()):
    print(f"Trang {node}: PageRank = {rank:.4f}")

print("Tổng PageRank:", sum(r for _, r in ranks_rdd.collect()))

sc.stop()

Kết quả PageRank sau 10 vòng lặp (không có damping factor):
Trang 0: PageRank = 0.4141
Trang 1: PageRank = 0.1953
Trang 2: PageRank = 0.3906
Trang 3: PageRank = 0.0000
Tổng PageRank: 1.0


In [5]:
from pyspark import SparkContext, SparkConf
# Khởi tạo Spark
conf = SparkConf().setAppName("PageRankWithDampingAndDeadEnds").setMaster("local[*]")
sc = SparkContext(conf=conf)
# Tổng số lượng node trong đồ thị (N = 5)
num_nodes = 5
# 1. Định nghĩa đồ thị gồm 5 node (0, 1, 2, 3, 4) dưới dạng danh sách các cạnh (source, target)
# Lưu ý: Node 4 trong ví dụ này có thể là một Dead End (không có cạnh đi ra)
edges_data = [
    (0, 1), (0, 2), # 0 trỏ tới 1, 2
    (1, 2),         # 1 trỏ tới 2
    (2, 0),         # 2 trỏ tới 0
    (3, 2), (3, 4)  # 3 trỏ tới 2, 4
    # Node 4 không trỏ đi đâu cả -> là Dead End
]

edges_rdd = sc.parallelize(edges_data)
# Danh sách đầy đủ tất cả các node để đảm bảo các dead node (không xuất hiện ở cột nguồn) vẫn được giữ lại
all_nodes_rdd = sc.parallelize(range(num_nodes)).map(lambda node: (node, []))
# Gom nhóm các cạnh theo node nguồn, sau đó kết hợp với tất cả các node để bao gồm cả dead nodes
# Kết quả: (node, [neighbors])
links_rdd = edges_rdd.groupByKey() \
    .rightOuterJoin(all_nodes_rdd) \
    .mapValues(lambda x: list(x[0]) if x[0] is not None else []) \
    .cache()

# 2. Khởi tạo PageRank ban đầu cho mỗi node bằng 1.0 / N
initial_rank = 1.0 / num_nodes
ranks_rdd = links_rdd.map(lambda x: (x[0], initial_rank))
damping_factor = 0.85 # Hệ số suy giảm (dampening factor)
num_iterations = 10
for iteration in range(num_iterations):
    # Bước a: Tách riêng phần PageRank bị rò rỉ từ các Dead Ends (nút có danh sách kề rỗng)
    # Join links_rdd và ranks_rdd -> (node, ([neighbors], rank))
    joined_for_dead_ends = links_rdd.join(ranks_rdd)

    # Tính tổng điểm PageRank của tất cả các dead ends
    dead_ends_sum = joined_for_dead_ends.filter(lambda x: len(x[1][0]) == 0) \
        .values() \
        .values() \
        .sum()

    # Bước b: Phân phối PageRank từ các nút bình thường tới các neighbors của chúng
    # Sử dụng flatMap để sinh ra các cặp (neighbor_node, contribution)
    contributions_rdd = joined_for_dead_ends.flatMap(lambda x:
        [(neighbor, x[1][1] / len(x[1][0])) for neighbor in x[1][0]] if len(x[1][0]) > 0 else [])
    # Bước c: Cộng dồn các đóng góp theo từng node đích bằng reduceByKey
    accumulated_ranks = contributions_rdd.reduceByKey(lambda a, b: a + b)
    # Bước d: Áp dụng công thức chuẩn của PageRank có damping factor và xử lý dead-end leak
    # Công thức mới cho mỗi node: PR(p) = (1 - d)/N + d * (sum(PR/deg) + dead_ends_sum / N)
    ranks_rdd = links_rdd.leftOuterJoin(accumulated_ranks) \
        .mapValues(lambda x: (1.0 - damping_factor) / num_nodes +
                             damping_factor * ((x[1] if x[1] is not None else 0.0) + (dead_ends_sum / num_nodes)))

# 3. Thu thập và hiển thị kết quả PageRank sau khi hội tụ
print(f"Kết quả PageRank sau {num_iterations} vòng lặp (d = {damping_factor}, có xử lý Dead Ends):")
total_rank = 0.0
for node, rank in sorted(ranks_rdd.collect()):
    print(f"Trang {node}: PageRank = {rank:.4f}")
    total_rank += rank
print(f"Tổng PageRank của toàn hệ thống: {total_rank:.4f}")
sc.stop()

Kết quả PageRank sau 10 vòng lặp (d = 0.85, có xử lý Dead Ends):
Trang 0: PageRank = 0.3513
Trang 1: PageRank = 0.1880
Trang 2: PageRank = 0.3647
Trang 3: PageRank = 0.0396
Trang 4: PageRank = 0.0564
Tổng PageRank của toàn hệ thống: 1.0000


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
from pyspark import SparkContext, SparkConf
# Khởi tạo SparkContext (nếu chưa có)
sc = SparkContext.getOrCreate()
# Đường dẫn đến file graph-small.txt trên Drive của bạn
file_path = "/content/drive/MyDrive/Bigdata/graph-small.txt"  # Thay đổi nếu file nằm trong thư mục con, VD: /content/d
# Đọc file thành RDD các dòng văn bản
graph_rdd = sc.textFile(file_path)
# Xem 5 dòng đầu tiên của file
print("5 dòng đầu tiên của RDD:")
for line in graph_rdd.take(5):
  print(line)

5 dòng đầu tiên của RDD:
100	1
13	1
28	1
89	1
82	1


In [8]:
from pyspark import SparkContext, SparkConf

# 1. Khởi tạo SparkContext
sc = SparkContext.getOrCreate()

# 2. Đọc file dữ liệu graph-small.txt
file_path = "/content/drive/MyDrive/Bigdata/graph-small.txt"  # Thay đổi đường dẫn file nếu dùng Drive: "/content/drive/MyDrive/data/graph-small.txt"
graph_rdd = sc.textFile(file_path)

# 3. Tiền xử lý dữ liệu đồ thị (tách dòng thành các cặp edge: source -> target)
# Giả định file phân tách bằng khoảng trắng hoặc tab
edges_rdd = graph_rdd.map(lambda line: line.strip().split()) \
                     .filter(lambda parts: len(parts) >= 2) \
                     .map(lambda parts: (int(parts[0]), int(parts[1])))

# Lấy danh sách tất cả các node duy nhất trong đồ thị
all_nodes = edges_rdd.flatMap(lambda edge: [edge[0], edge[1]]).distinct()
num_nodes = all_nodes.count()

# Tạo danh sách danh sách kề (links) bao gồm cả Dead Ends (nút không có cạnh đi ra)
all_nodes_rdd = all_nodes.map(lambda node: (node, []))
links_rdd = edges_rdd.groupByKey() \
                     .rightOuterJoin(all_nodes_rdd) \
                     .mapValues(lambda x: list(x[0]) if x[0] is not None else []) \
                     .cache()

# 4. Khởi tạo PageRank ban đầu = 1.0 / N
initial_rank = 1.0 / num_nodes
ranks_rdd = links_rdd.map(lambda x: (x[0], initial_rank))

# Cấu hình tham số
beta = 1.0  # beta = 1 (không suy giảm)
num_iterations = 10

# 5. Vòng lặp tính PageRank
for iteration in range(num_iterations):
    # Join links và ranks -> (node, ([neighbors], rank))
    joined_rdd = links_rdd.join(ranks_rdd)

    # Tính tổng PageRank rò rỉ từ các Dead Ends
    dead_ends_sum = joined_rdd.filter(lambda x: len(x[1][0]) == 0) \
                              .map(lambda x: x[1][1]) \
                              .sum()

    # Phân phối PageRank tới các nút kề (neighbors)
    contributions_rdd = joined_rdd.flatMap(
        lambda x: [(neighbor, x[1][1] / len(x[1][0])) for neighbor in x[1][0]]
        if len(x[1][0]) > 0 else []
    )

    # Cộng dồn các đóng góp và tính rank mới với beta = 1
    # Công thức: PR(p) = beta * (sum(PR/deg) + dead_ends_sum / N)
    accumulated_ranks = contributions_rdd.reduceByKey(lambda a, b: a + b)
    ranks_rdd = links_rdd.leftOuterJoin(accumulated_ranks) \
        .mapValues(lambda x: beta * ((x[1] if x[1] is not None else 0.0) + (dead_ends_sum / num_nodes)))

# 6. Hiển thị kết quả PageRank
print(f"--- Kết quả PageRank sau {num_iterations} vòng lặp (beta = {beta}) ---")
total_rank = 0.0
for node, rank in sorted(ranks_rdd.collect()):
    print(f"Trang {node}: PageRank = {rank:.6f}")
    total_rank += rank

print(f"Tổng PageRank của toàn hệ thống: {total_rank:.6f}")

--- Kết quả PageRank sau 10 vòng lặp (beta = 1.0) ---
Trang 1: PageRank = 0.042082
Trang 2: PageRank = 0.004720
Trang 3: PageRank = 0.006908
Trang 4: PageRank = 0.006320
Trang 5: PageRank = 0.015211
Trang 6: PageRank = 0.004920
Trang 7: PageRank = 0.002320
Trang 8: PageRank = 0.007859
Trang 9: PageRank = 0.013923
Trang 10: PageRank = 0.007666
Trang 11: PageRank = 0.003869
Trang 12: PageRank = 0.003380
Trang 13: PageRank = 0.019105
Trang 14: PageRank = 0.041754
Trang 15: PageRank = 0.005921
Trang 16: PageRank = 0.003962
Trang 17: PageRank = 0.004330
Trang 18: PageRank = 0.013345
Trang 19: PageRank = 0.004517
Trang 20: PageRank = 0.003463
Trang 21: PageRank = 0.005331
Trang 22: PageRank = 0.015735
Trang 23: PageRank = 0.003190
Trang 24: PageRank = 0.004841
Trang 25: PageRank = 0.005662
Trang 26: PageRank = 0.014834
Trang 27: PageRank = 0.039508
Trang 28: PageRank = 0.006284
Trang 29: PageRank = 0.005683
Trang 30: PageRank = 0.005449
Trang 31: PageRank = 0.011941
Trang 32: PageRank = 0.00

In [9]:
from pyspark import SparkContext

# 1. Khởi tạo SparkContext
sc = SparkContext.getOrCreate()

# 2. Đọc file dữ liệu (Sử dụng đường dẫn Drive hoặc file trực tiếp)
# Nếu file nằm trên Drive:
file_path = "/content/drive/MyDrive/Bigdata/graph-small.txt"
#file_path = "graph-small.txt"
graph_rdd = sc.textFile(file_path)

# 3. Tiền xử lý dữ liệu đồ thị
edges_rdd = graph_rdd.map(lambda line: line.strip().split()) \
                     .filter(lambda parts: len(parts) >= 2) \
                     .map(lambda parts: (int(parts[0]), int(parts[1])))

# Lấy tất cả các nút duy nhất trong đồ thị
all_nodes = edges_rdd.flatMap(lambda edge: [edge[0], edge[1]]).distinct()
num_nodes = all_nodes.count()

# Tạo danh sách kề bao gồm cả Dead Ends (nút không có cạnh đi ra)
all_nodes_rdd = all_nodes.map(lambda node: (node, []))
links_rdd = edges_rdd.groupByKey() \
                     .rightOuterJoin(all_nodes_rdd) \
                     .mapValues(lambda x: list(x[0]) if x[0] is not None else []) \
                     .cache()

# 4. Khởi tạo PageRank ban đầu = 1.0 / N
initial_rank = 1.0 / num_nodes
ranks_rdd = links_rdd.map(lambda x: (x[0], initial_rank))

# Cấu hình tham số BÀI 7 (beta = 0.85)
beta = 0.85
num_iterations = 10

# 5. Vòng lặp tính PageRank có Damping Factor (beta = 0.85)
for iteration in range(num_iterations):
    joined_rdd = links_rdd.join(ranks_rdd)

    # Tính tổng PageRank rò rỉ từ các Dead Ends
    dead_ends_sum = joined_rdd.filter(lambda x: len(x[1][0]) == 0) \
                              .map(lambda x: x[1][1]) \
                              .sum()

    # Phân phối PageRank tới các nút kề
    contributions_rdd = joined_rdd.flatMap(
        lambda x: [(neighbor, x[1][1] / len(x[1][0])) for neighbor in x[1][0]]
        if len(x[1][0]) > 0 else []
    )

    # Công thức PageRank chuẩn: PR(p) = (1 - beta)/N + beta * (sum(PR/deg) + dead_ends_sum / N)
    accumulated_ranks = contributions_rdd.reduceByKey(lambda a, b: a + b)
    ranks_rdd = links_rdd.leftOuterJoin(accumulated_ranks) \
        .mapValues(lambda x: (1.0 - beta) / num_nodes +
                             beta * ((x[1] if x[1] is not None else 0.0) + (dead_ends_sum / num_nodes)))

# 6. Hiển thị kết quả BÀI 7
print(f"--- BÀI 7: Kết quả PageRank với beta = {beta} ---")
total_rank = 0.0
for node, rank in sorted(ranks_rdd.collect()):
    print(f"Trang {node}: PageRank = {rank:.6f}")
    total_rank += rank

print(f"Tổng PageRank của toàn hệ thống: {total_rank:.6f}")

--- BÀI 7: Kết quả PageRank với beta = 0.85 ---
Trang 1: PageRank = 0.036859
Trang 2: PageRank = 0.005296
Trang 3: PageRank = 0.007346
Trang 4: PageRank = 0.006879
Trang 5: PageRank = 0.014602
Trang 6: PageRank = 0.005779
Trang 7: PageRank = 0.003626
Trang 8: PageRank = 0.008063
Trang 9: PageRank = 0.013652
Trang 10: PageRank = 0.007763
Trang 11: PageRank = 0.004660
Trang 12: PageRank = 0.004674
Trang 13: PageRank = 0.017465
Trang 14: PageRank = 0.037374
Trang 15: PageRank = 0.006235
Trang 16: PageRank = 0.005280
Trang 17: PageRank = 0.005512
Trang 18: PageRank = 0.013353
Trang 19: PageRank = 0.005223
Trang 20: PageRank = 0.004512
Trang 21: PageRank = 0.005983
Trang 22: PageRank = 0.014539
Trang 23: PageRank = 0.004144
Trang 24: PageRank = 0.005909
Trang 25: PageRank = 0.006519
Trang 26: PageRank = 0.014233
Trang 27: PageRank = 0.034692
Trang 28: PageRank = 0.006511
Trang 29: PageRank = 0.006483
Trang 30: PageRank = 0.006369
Trang 31: PageRank = 0.012081
Trang 32: PageRank = 0.004438
T